# PlanTo3D — a floor plan, your way

Upload a plan, choose how the house should look, get a model and six
rendered views. The photoreal pass at the end needs a GPU; everything
before it does not.

**Set the runtime to GPU first:** Runtime → Change runtime type → T4 GPU.

## What you choose, and what the drawing decides

The drawing fixes the geometry: where the walls are, how the rooms divide,
where the doors and windows sit. None of that is a choice, and none of it
is offered here — the output is meant to be the plan, in three dimensions.

What a drawing never says is what the building is *made* of, what hour it
is seen at, or whether there is a garden. Those are five choices in
section 4.


## 1. Setup

In [ ]:
!apt-get -qq install -y poppler-utils tesseract-ocr > /dev/null
!pip install -q trimesh shapely mapbox-earcut pytesseract pdf2image \
    segmentation-models-pytorch diffusers transformers accelerate safetensors
print("dependencies ready")

In [ ]:
import sys
from pathlib import Path

repo = Path("/content/PlanTo3D")
if repo.exists():
    !cd {repo} && git pull --quiet
else:
    !git clone --quiet https://github.com/priyanshsoni096-blip/PlanTo3D.git {repo}

if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))
%cd {repo}

import torch

print(
    f"GPU: {torch.cuda.get_device_name(0)}"
    if torch.cuda.is_available()
    else "no GPU -- everything but the photoreal pass will still run"
)

## 2. The trained segmenter

Upload the checkpoint, or mount Drive if you keep it there. Without one the
classical baseline is used, and it only reads cleanly drafted CAD sheets.

In [ ]:
from google.colab import drive

models = Path("/content/PlanTo3D/models")
models.mkdir(exist_ok=True)

drive.mount("/content/drive")
checkpoint = Path("/content/drive/MyDrive/planto3d/unet_cubicasa.pt")

if checkpoint.is_file():
    !cp {checkpoint} {models}/unet_cubicasa.pt
    print(f"using {checkpoint}")
else:
    print("no checkpoint on Drive -- upload one below, or the baseline is used")
    from google.colab import files

    for name in files.upload():
        Path(name).rename(models / name)

## 3. The floor plan

**Upload the original PDF rather than a screenshot.** Room names drive the
floor finishes, planting and railings, and reading them needs resolution.

One file per storey, ground floor first, or a single multi-page PDF.

In [ ]:
from google.colab import files

uploaded = sorted(files.upload())
plans = Path("/content/plan")
plans.mkdir(exist_ok=True)

for index, name in enumerate(uploaded):
    Path(name).rename(plans / f"{index:02d}{Path(name).suffix.lower()}")

SOURCE = plans if len(uploaded) > 1 else plans / sorted(p.name for p in plans.iterdir())[0]
print(f"{len(uploaded)} file(s): {SOURCE}")

## 4. How it should look

Five choices. Run the cell after changing any of them.

**Style** is what the building is made of. **Colour** takes that lighter,
darker or warmer. **Time** is the hour. **Landscaping** decides how much of
a setting it gets -- `none` leaves it alone against the sky, which is what
a massing study wants and a presentation render does not. **Creativity**
only affects the photoreal pass in section 7: strict holds the geometry and
looks increasingly like a shaded model, creative invents freely and stops
describing this particular house.

In [ ]:
#@title Choose { run: "auto" }
style = "luxury"  #@param ["modern", "luxury", "traditional", "minimalist"]
colour = "warm"  #@param ["light", "dark", "warm"]
time = "day"  #@param ["day", "sunset", "night"]
landscaping = "premium"  #@param ["none", "basic", "premium"]
creativity = "balanced"  #@param ["strict", "balanced", "creative"]
storey_height_ft = 9  #@param {type:"slider", min:7, max:14, step:0.5}

from planto3d.design import Design

DESIGN = Design(
    style=style,
    colour=colour,
    time=time,
    landscaping=landscaping,
    creativity=creativity,
)
print(DESIGN)

## 5. Build it

Reads the drawing, measures it, and extrudes the model. A minute or two.

In [ ]:
import logging

from planto3d.pipeline import run
from planto3d.segment import load_segmenter

logging.basicConfig(level=logging.WARNING, format="%(levelname)s %(message)s")

OUT = Path("/content/output")
result = run(
    SOURCE,
    OUT,
    segmenter=load_segmenter(next(iter(sorted(models.glob("*.pt"))), None)),
    wall_height_ft=storey_height_ft,
    palette=DESIGN.palette(),
    site=DESIGN.site(),
)

print(f"{len(result.floors)} storey(s), {result.wall_count} walls, {result.room_count} rooms")
print(f"scale {result.scale:.1f} px/ft, from {result.scale_source}")
for floor in result.floors:
    names = ", ".join(floor.named_rooms) or "no names read"
    print(f"  floor {floor.index + 1}: {names}")

if result.scale_assumed:
    print("\nProportions are right; the absolute size is inferred.")

## 6. Look at it

Six views. The rendered image is what the building actually looks like --
an interactive viewer over-lights masonry and cannot be told not to.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

from planto3d.preview import render_views

views = render_views(
    result.model_path, OUT, resolution=(1100, 820), lighting=DESIGN.lighting()
)

order = ["aerial", "front", "back", "left", "right", "top"]
figure, axes = plt.subplots(2, 3, figsize=(21, 11))
for axis, name in zip(axes.ravel(), order):
    axis.imshow(Image.open(views[name]))
    axis.set_title(name, fontsize=13)
    axis.axis("off")
plt.tight_layout()
plt.show()

## 7. The photoreal pass

Diffusion, conditioned on the model's depth so it dresses this building
rather than inventing another. Needs the GPU.

The prompt is built from what the pipeline actually read off the drawing --
cars where it found parking, lawn where it found planting -- and the
conditioning comes from the creativity chosen in section 4.

In [ ]:
from diffusers import (
    ControlNetModel,
    StableDiffusionControlNetPipeline,
    UniPCMultistepScheduler,
)

if not torch.cuda.is_available():
    raise SystemExit("No GPU. Runtime -> Change runtime type -> T4.")

controlnet = ControlNetModel.from_pretrained(
    "lllyasviel/control_v11f1p_sd15_depth", torch_dtype=torch.float16
)
PIPE = StableDiffusionControlNetPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    controlnet=controlnet,
    torch_dtype=torch.float16,
    safety_checker=None,
)
PIPE.scheduler = UniPCMultistepScheduler.from_config(PIPE.scheduler.config)
PIPE = PIPE.to("cuda")
PIPE.enable_attention_slicing()
print("photoreal pass ready")

In [ ]:
from planto3d.photoreal import NEGATIVE_PROMPT, build_guides, build_prompt

guides = build_guides(result.model_path, OUT / "guides")
depth = Image.open(guides["depth"]).convert("RGB")

# SD 1.5 is happiest near 768, and works in multiples of eight.
longest = max(depth.size)
if longest > 768:
    depth = depth.resize(tuple(int(d * 768 / longest) for d in depth.size))
width, height = (max(d - d % 8, 8) for d in depth.size)

labels = [label for floor in result.floors for label in floor.named_rooms]
prompt = build_prompt(len(result.floors), labels)
print(f"{width}x{height}, conditioning {DESIGN.conditioning()}\n\n{prompt}")

image = PIPE(
    prompt=prompt,
    negative_prompt=NEGATIVE_PROMPT,
    image=depth.resize((width, height)),
    num_inference_steps=30,
    guidance_scale=8.0,
    controlnet_conditioning_scale=DESIGN.conditioning(),
    generator=torch.Generator(device="cuda").manual_seed(7),
).images[0]

image.save(OUT / "photoreal.png")
plt.figure(figsize=(13, 10))
plt.imshow(image)
plt.axis("off")
plt.show()

## 8. Take it home

The `.glb` opens in Windows 3D Viewer, Blender, or anything that reads
glTF.

In [ ]:
import shutil

shutil.make_archive("/content/planto3d_output", "zip", OUT)
files.download("/content/planto3d_output.zip")